In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load

def random_date(start_date, end_date):
    """Generate a random date between start_date and end_date."""
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=random_days)

def compute_Landsat_values(lat, lon, date):
    bbox_size = 0.00089831
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2015-01-01/2023-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
   
    items = search.item_collection()
    if not items:
        return None
    try:
        sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
        )
        selected_item = pc.sign(items[0])
        bands_of_interest = ["green", "nir08", "swir16", "swir22"]
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        green = data["green"].astype("float")
        nir = data["nir08"].astype("float")
        swir16 = data["swir16"].astype("float")
        swir22 = data["swir22"].astype("float")
        median_green = float(green.median(skipna=True).values)
        median_nir = float(nir.median(skipna=True).values)
        median_swir16 = float(swir16.median(skipna=True).values)
        median_swir22 = float(swir22.median(skipna=True).values)
        median_green = median_green if median_green != 0 else np.nan
        median_nir = median_nir if median_nir != 0 else np.nan
        median_swir16 = median_swir16 if median_swir16 != 0 else np.nan
        median_swir22 = median_swir22 if median_swir22 != 0 else np.nan
       
        if np.isnan(median_nir) or np.isnan(median_green) or np.isnan(median_swir16) or np.isnan(median_swir22):
            return None
        
        return {
            "nir": median_nir,
            "green": median_green,
            "swir16": median_swir16,
            "swir22": median_swir22
        }
   
    except Exception as e:
        return None

np.random.seed(42) 
num_samples_needed = 200
samples = []
start_date = datetime(2015, 1, 1)
end_date = datetime(2023, 12, 31)

pbar = tqdm(total=num_samples_needed, desc="Collecting valid samples")
attempts = 0
while len(samples) < num_samples_needed:
    attempts += 1
    lat = np.random.uniform(-90, 90)
    lon = np.random.uniform(-180, 180)
    date = random_date(start_date, end_date)
    
    features = compute_Landsat_values(lat, lon, date)
    if features:
        samples.append({
            'Latitude': lat,
            'Longitude': lon,
            'Sample Date': date,
            **features
        })
        pbar.update(1)

pbar.close()
print(f"Total attempts: {attempts}")

df_with_features = pd.DataFrame(samples)
eps = 1e-10
df_with_features['NDMI'] = (df_with_features['nir'] - df_with_features['swir16']) / (df_with_features['nir'] + df_with_features['swir16'] + eps)
df_with_features['MNDWI'] = (df_with_features['green'] - df_with_features['swir16']) / (df_with_features['green'] + df_with_features['swir16'] + eps)
df_with_features.to_csv('landsat_features_2015_2023_200_samples.csv', index=False)

KeyboardInterrupt: 